# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmoeed1090/Fly-rank-starter-assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 1. My Rule and its Reason Codes

## Baseline Rule

The baseline assigns every content page an opportunity score using only the current Search Console metrics. The goal is to identify pages that are likely to benefit from a content refresh or metadata improvement.

The rule uses three simple signals:

- Low CTR despite having impressions → **LOW_CTR**
- Poor average search position → **LOW_POSITION**
- Otherwise → **MONITOR**

Each reason code is mapped to an action:

| Reason Code | Action |
|-------------|-------------------------------|
| LOW_CTR | Review Title & Meta Description |
| LOW_POSITION | Refresh Content |
| MONITOR | Monitor |

This rule uses only information available before making the decision. It does not use future information or label-derived columns, making it a fair baseline for comparison with future machine learning models.

In [12]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
import pandas as pd

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

df = dataset["train"].to_pandas()

# Use first 50k rows to keep notebook responsive
df = df.head(50000)

print(df.shape)
df.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

(50000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30.0,0.0,115.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5.0,0.0,358.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1.0,0.0,34.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6.0,0.0,140.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5.0,0.0,89.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Build the ranked queue (writes the CSV)


The baseline score is created using three search signals.

Scoring rules:

- Higher impressions increase opportunity.
- Lower CTR increases opportunity.
- Poor average search position increases opportunity.

After calculating the score, every page receives a reason code and an action recommendation.

Finally, all pages are ranked from highest opportunity to lowest opportunity.

In [16]:
# Fill missing values
df["gsc_impressions"] = df["gsc_impressions"].fillna(0)
df["gsc_clicks"] = df["gsc_clicks"].fillna(0)
df["gsc_avg_position"] = df["gsc_avg_position"].fillna(0)

# CTR
df["ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"].replace(0,1)
)

# Baseline Score
df["baseline_score"] = (
    df["gsc_impressions"]*0.4 +
    (1-df["ctr"])*40 +
    df["gsc_avg_position"]*0.2
)

# Reason Codes
def reason(row):

    if row["ctr"] < 0.02 and row["gsc_impressions"] > 20:
        return "LOW_CTR"

    elif row["gsc_avg_position"] > 30:
        return "LOW_POSITION"

    else:
        return "MONITOR"

df["reason_code"] = df.apply(reason, axis=1)

# Actions
def get_action(reason):

    if reason=="LOW_CTR":
        return "Review Title & Meta Description"

    elif reason=="LOW_POSITION":
        return "Refresh Content"

    else:
        return "Monitor"

df["action"] = df["reason_code"].apply(get_action)

# Ranking
ranked_df = df.sort_values(
    by="baseline_score",
    ascending=False
)

ranked_df["rank"] = range(1,len(ranked_df)+1)

ranked_df.head()

import os

os.makedirs("work/outputs", exist_ok=True)

ranked_df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully.")


CSV saved successfully.


## 3. Top-20 review

The table below shows the twenty pages with the highest baseline scores.

For each page, the baseline provides:

- Opportunity score
- Reason code
- Recommended action

These recommendations are decision-support only. Human review is still required before updating any content.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked_df[
    [
        "rank",
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
].head(20)

top20

,rank,content_hash_id,baseline_score,reason_code,action
38579,1,content_690b092cf66bc2a4,366.875061,LOW_CTR,Review Title & Meta Description
42839,2,content_690b092cf66bc2a4,336.307278,MONITOR,Monitor
47089,3,content_690b092cf66bc2a4,325.202241,LOW_CTR,Review Title & Meta Description
34266,4,content_690b092cf66bc2a4,275.824407,LOW_CTR,Review Title & Meta Description
14715,5,content_a64143f6e4a21ffe,272.445172,LOW_CTR,Review Title & Meta Description
12108,6,content_06de5368fbd3bf99,258.401121,LOW_CTR,Review Title & Meta Description
41328,7,content_4d067c48a4f50221,254.983698,LOW_CTR,Review Title & Meta Description
21286,8,content_690b092cf66bc2a4,252.826592,MONITOR,Monitor
12435,9,content_690b092cf66bc2a4,252.339474,MONITOR,Monitor
20971,10,content_06de5368fbd3bf99,244.732000,LOW_CTR,Review Title & Meta Description


# 4. Weak Picks and Leakage Check

Some pages may receive high opportunity scores even though they do not actually need a refresh. For example, seasonal content or newly published pages may naturally have lower performance.

To avoid leakage, only current Search Console metrics are used. No future outcomes, label-derived columns, or product-generated flags are included in the baseline score.

This baseline is intended as a transparent decision-support rule that future machine learning models should improve upon.

In [18]:
print("Reason Code Distribution")
print(df["reason_code"].value_counts())

print()

print("Action Distribution")
print(df["action"].value_counts())

print()

excluded = [
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

print("Excluded columns:")
print(excluded)

print()

print("Leakage Check Passed")
print("No future columns were used.")

Reason Code Distribution
reason_code
MONITOR         24935
LOW_POSITION    15472
LOW_CTR          9593
Name: count, dtype: int64

Action Distribution
action
Monitor                            24935
Refresh Content                    15472
Review Title & Meta Description     9593
Name: count, dtype: int64

Excluded columns:
['client_hash_id', 'content_hash_id', 'report_date']

Leakage Check Passed
No future columns were used.


In [11]:
print(df["reason_code"].value_counts())

reason_code
MONITOR         34144
LOW_POSITION    13349
LOW_CTR          2507
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.